E-12 (main table): exact balance, inference, omitted direction, BP existence

Registration: `doc/2026-10-07_experiment_registration.md` (parent repository), E-12 and §1.4–§1.9; Amendment 4 for the Newton acceptance. One notebook for every stage; the runner passes the stage:

- `python3 notebooks/experiments/run_registered.py E-12 stage0 10_E12_omitted_direction_balance.ipynb`
- `... E-12 stage0.5 ...`, then `... E-12 stage1 ...` (each after the previous stage is committed).

DGP, arms, the replication function and the aggregation are in `grrexp/e12.py`; the pilot and the confirmatory stage run the same aggregation. A solver that raises stops the notebook and the run is recorded as failed (§1.1).

In [1]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import baselines, e12, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
rec = outputs.RunRecorder(12, STAGE, cells=None if STAGE == "stage0" else e12.CELLS,
                          arms=None if STAGE == "stage0" else [a.replace("|", "-") for a in e12.ARM_LABELS])
STAGE

'stage0'

# Stage 0: population quantities (registration E-12 Stage 0, §1.9)

In [2]:
if STAGE == "stage0":
    (rows, gate), stage0_warnings, _ = baselines.run_recording_warnings(e12.stage0)
    rec.write_stage0("e12_population", {"cells": rows, "reference_gate": gate})
    print(json.dumps(gate, indent=1))
    print({(r["arm"], r["set"], r["s"]): r["certified"] for r in rows})
    bad = [k for k, v in gate.items() if not v["ok"]]
    if bad:
        raise RuntimeError(f"Stage 0 reference gate failed: {bad}")  # registration: stop
    rec.finalize(R=None, n=None, warnings=int(sum(stage0_warnings.values())), failures=0)

{
 "UKL1/Omit/1.25/sigma2_true": {
  "computed": 7.290654874685092,
  "reference": 7.2906549,
  "ok": true
 },
 "UKL1/Omit/1.25/sigma2_se": {
  "computed": 7.601303231555001,
  "reference": 7.6013032,
  "ok": true
 },
 "SQ/Include/0.5/E_alpha2": {
  "computed": 4.366446584141031,
  "reference": 4.3664466,
  "ok": true
 },
 "UKL1/Include/0.5/E_alpha2": {
  "computed": 4.417336924549454,
  "reference": 4.4173369,
  "ok": true
 }
}
{('SQ', 'Include', 0.5): True, ('SQ', 'Include', 1.25): True, ('SQ', 'Omit', 0.5): True, ('SQ', 'Omit', 1.25): True, ('UKL1', 'Include', 0.5): True, ('UKL1', 'Include', 1.25): True, ('UKL1', 'Omit', 0.5): True, ('UKL1', 'Omit', 1.25): True, ('BKL1', 'Include', 0.5): True, ('BKL1', 'Include', 1.25): False, ('BKL1', 'Omit', 0.5): True, ('BKL1', 'Omit', 1.25): False, ('BP_C1', 'Include', 0.5): True, ('BP_C1', 'Include', 1.25): False, ('BP_C1', 'Omit', 0.5): True, ('BP_C1', 'Omit', 1.25): False, ('BP_C0', 'Include', 0.5): True, ('BP_C0', 'Include', 1.25): True, ('B

# The registered workflow after the replications (shared by Stage 0.5 and Stage 1)

H12-Bal (stops on a violation), the §1.5 metrics, the family-1 MCSE intervals, the families H12-Inf, H12-Bias, H12-EF, H12-BP (Holm 0.01) and H12-Ord.

In [3]:
def load_population():
    return json.loads((outputs.stage_dir(12, "stage0") / "stage0_e12_population.json").read_text())["cells"]


def aggregate(raw, pop, seeds):
    bal = e12.balance_checks(raw)
    summ = e12.summarise(raw, pop)
    summ = summ.assign(**e12.family1_mcse(raw, summ, seeds))
    fam, unavailable = e12.families(summ, raw)
    ord_cell = e12.CELLS.index([1.25, "Include", 4000])
    ordr = e12.ordering(raw, seeds) if (raw["cell"] == ord_cell).any() else None
    summ["family_verdicts"] = json.dumps({k: (v.sentence() if v else None) for k, v in fam.items()})
    summ["unavailable_tests"] = json.dumps(unavailable)
    summ["ord"] = json.dumps(ordr)
    summ["h12_bal"] = json.dumps(bal)
    return summ

# Stage 0.5: timing of the complete workflow, status counts, 1-versus-12-worker identity (no estimates kept)

Per cell and number of workers: the GBM tuning of $(s, n)$ (timed with its Include cell, where it is done), the 10 replications, and the aggregation of the cell. The aggregation of the whole pilot sample and the table generation are timed once and recorded in the manifest. Pilot estimates and tables are discarded.

In [4]:
if STAGE == "stage0.5":
    parallel.configure_worker()
    pop = load_population()
    timing, serial, para, all_tasks, gbm, tune_warnings = [], [], [], [], {}, 0
    for workers in (1, outputs.PILOT_WORKERS):
        seeds = Seeds(12, entropy=PILOT_ENTROPY)
        for ci, (s, rs, n) in enumerate(e12.CELLS):
            t0 = time.perf_counter()
            if rs == "Include":
                gbm[(s, n)], w = e12.tune_cell(seeds, ci)
                tune_warnings += w if workers == 1 else 0
            tasks = e12.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, gbm, cells=[ci])
            res = parallel.run_tasks(e12.replicate, tasks, workers)
            aggregate(e12.raw_frame(tasks, res), pop, Seeds(12, entropy=PILOT_ENTROPY))
            timing.append({"cell": e12.CELLS[ci], "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e12.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    buf = io.StringIO()
    aggregate(raw, pop, Seeds(12, entropy=PILOT_ENTROPY)).to_csv(buf, index=False)
    buf.seek(0)
    e12.tables(pd.read_csv(buf))
    whole_seconds = time.perf_counter() - t0
    counts = raw.groupby(["cell", "arm", "status"]).size().reset_index(name="count")
    status_df = pd.DataFrame({"cell": [e12.CELLS[c] for c in counts["cell"]],
                              "arm": counts["arm"].str.replace("|", "-", regex=False),
                              "status": counts["status"], "count": counts["count"].astype(int)})
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e12.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    rec.finalize(R={"main": outputs.PILOT_REPS}, n={"main": list(e12.N_VALUES)},
                 warnings=e12.total_warnings(raw) + tune_warnings,
                 failures=int((raw["status"] != "ok").sum()),
                 extra={"whole_pilot_aggregation_and_tables_seconds": whole_seconds,
                        "timing_note": "the GBM tuning of (s, n) is timed with its Include cell"})
    del raw

# Stage 1: confirmatory Monte Carlo

In [5]:
if STAGE == "stage1":
    parallel.configure_worker()
    gbm, tune_warnings = e12.tune_all(CONFIRMATORY_ENTROPY)
    tasks = e12.tasks_for(CONFIRMATORY_ENTROPY, e12.R, gbm)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e12.replicate, tasks, outputs.PILOT_WORKERS)
    print("seconds", time.perf_counter() - t0)
    raw = e12.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn = e12.total_warnings(raw) + tune_warnings
    raw.shape, n_warn

In [6]:
if STAGE == "stage1":
    summ = aggregate(raw, load_population(), Seeds(12))  # H12-Bal stops the notebook on a violation
    rec.write_summary(summ)
    {k: json.loads(summ[k].iloc[0]) for k in ("h12_bal", "family_verdicts", "unavailable_tests", "ord")}

# Tables and macros (from summary.csv only, §4)

In [7]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    tabs, macros = e12.tables(S)
    for name, tex in tabs.items():
        rec.write_table(name, tex)
    rec.write_macros(macros)
    rec.finalize(R={"main": e12.R}, n={"main": list(e12.N_VALUES)}, warnings=int(n_warn),
                 failures=int((S["R"] - S["R_s"]).sum()))